# E04 · Observability, complementary filters and the extended Kalman filter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E04_observability_complementary_filters_and_the_exte.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 60 min

**Skill Hub concepts:** Observability; Complementary filter; Extended Kalman filter; Jacobian

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E04")

## Map: can you estimate it at all, and what if the model is nonlinear?

**Observability (linear, time-invariant):** the state can be reconstructed from outputs iff
$\mathcal{O} = \begin{bmatrix} H \\ HF \\ HF^2 \\ \vdots \\ HF^{n-1}\end{bmatrix}$ has rank $n$.
Measuring only velocity never reveals absolute position: rank deficient.

**Complementary filter (tilt from an IMU):** gyro integration is smooth but drifts; the accelerometer angle is
drift-free but noisy and disturbed by motion. Blend high-pass gyro with low-pass accel:
$$\theta_k = \alpha(\theta_{k-1} + \omega_k \Delta t) + (1 - \alpha)\,\theta_{acc,k}, \qquad \alpha = \frac{\tau}{\tau + \Delta t}$$

**Extended Kalman filter:** same equations as lab E03, but with nonlinear $f$ and $h$, using their **Jacobians**
evaluated at the current estimate:
$$y = z - h(\hat x), \quad H = \frac{\partial h}{\partial x}\Big|_{\hat x}, \quad S = HPH^\top + R, \quad K = PH^\top S^{-1}$$

Range and bearing to a landmark at $(l_x, l_y)$ from position $(x, y)$, with $d_x = l_x - x$, $d_y = l_y - y$,
$q = d_x^2 + d_y^2$:
$$h = \begin{bmatrix}\sqrt{q} \\ \operatorname{atan2}(d_y, d_x)\end{bmatrix}, \qquad
H = \begin{bmatrix} -d_x/\sqrt q & -d_y/\sqrt q \\ d_y/q & -d_x/q \end{bmatrix}$$
Always wrap bearing innovations to $[-\pi, \pi)$.

In [ ]:
import numpy as np

## Task 1: observability rank

In [ ]:
def observability_rank(F, H):
    raise NotImplementedError  # TODO

lab.check("observability", observability_rank)

## Task 2: complementary filter for pitch
Inputs per step: gyro rate (rad/s, with bias) and the accelerometer pitch angle (rad, noisy). Return the list of estimates.

In [ ]:
def complementary(gyro, acc_angle, dt, tau):
    raise NotImplementedError  # TODO

lab.check("complementary", complementary)

## Task 3: an EKF with range and bearing to one landmark
State: 2D position. Motion: random walk (F = I, Q given). Measurement: range and bearing.

In [ ]:
def h_meas(x, lm):
    dx, dy = lm[0] - x[0], lm[1] - x[1]
    return np.array([np.hypot(dx, dy), np.arctan2(dy, dx)])

def H_jac(x, lm):
    raise NotImplementedError  # TODO

def ekf_step(x, P, z, lm, Q, R):
    raise NotImplementedError  # TODO

lab.check("ekf", H_jac, ekf_step)

## Checkpoint: Math
**M1 (Apply).** Complementary filter with $\tau = 0.5$ s at 100 Hz. $\alpha$ (4 decimals)?

**M2 (Apply).** Constant-velocity model measured by velocity only. Rank of the observability matrix?

**M3 (Apply).** Robot at (0, 0), landmark at (3, 4). $\partial(\text{range})/\partial l_x$, i.e. $d_x/\sqrt{q}$?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why does a complementary filter beat either sensor alone for tilt?
- a) it averages them
- b) the gyro is trusted at high frequency (no drift there) and the accelerometer at low frequency (no drift long-term); each covers the other's weakness
- c) it removes the need for calibration

**K2 (Analyze).** When does an EKF fail, and what are the usual remedies?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e04.h`).
3. In `roboprep/checks/e04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()